# Data Cleaning with Pandas

Real-world data is almost never ready to analyze as-is. Sensors fail, spreadsheets have typos, columns get misformatted, and values go missing. **Data cleaning** is the process of detecting and fixing these problems so your analysis is built on a solid foundation.

In this notebook, we'll work through the most common data cleaning tasks:

1. **Inspecting** your data to understand its structure
2. **Fixing data types** so columns are stored correctly
3. **Handling missing data** — detecting, dropping, or filling it in
4. **Finding duplicates**
5. **Detecting outliers**
6. **Validating** that your cleaned data makes sense

We'll use a dataset of flight records from 2013, which is split across multiple files and has many of the issues you'll encounter in scientific data.

---
## Setup: Loading the data

The dataset is stored as 12 CSV files (one per month) in a GitHub repository. We'll clone it and combine the files into a single DataFrame. This is review from the data loading notebook — here we'll provide the code so we can focus on cleaning.

In [ ]:
import os
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

In [ ]:
# Clone the dataset from GitHub (12 CSV files, one per month)
!git clone https://github.com/eunjaeshim/raw_flights_data.git

In [ ]:
# Read all CSVs and combine into one DataFrame
all_dfs = []
for filename in sorted(os.listdir("raw_flights_data")):
    if filename.endswith(".csv"):
        df_part = pd.read_csv(os.path.join("raw_flights_data", filename), index_col=0)
        all_dfs.append(df_part)

flights = pd.concat(all_dfs, ignore_index=True)
print(f"Loaded {len(all_dfs)} files → {flights.shape[0]:,} rows, {flights.shape[1]} columns")

In [ ]:
flights.head()

---
## 1. Inspecting your data

Before cleaning anything, you need to understand what you're working with. There are three commands you should **always** run first on a new dataset:

### `.info()` — the big picture

[`.info()`](https://pandas.pydata.org/docs/reference/api/pandas.DataFrame.info.html) shows you column names, how many non-null values each has, and the **data type** (`dtype`) of each column. This is often the single most useful command for spotting problems.

In [ ]:
flights.info()

Things to look for:
- **Non-Null Count**: if a column has fewer non-null values than the total rows, it has missing data.
- **Dtype**: are numeric columns actually stored as numbers (`int64`, `float64`), or did something go wrong and they're stored as `object` (text)?

### `.describe()` — summary statistics

[`.describe()`](https://pandas.pydata.org/docs/reference/api/pandas.DataFrame.describe.html) gives you the count, mean, standard deviation, min, max, and quartiles for every numeric column. Scan this for anything that looks wrong — values that are physically impossible, unexpectedly large ranges, or suspicious patterns.

In [ ]:
flights.describe()

### `.dtypes` — a quick check

For a faster view of just the data types:

In [ ]:
flights.dtypes

### Exercise 1

Look at the output of `.info()` and `.describe()` above and answer these questions in the markdown cell below:
1. Which columns have missing values? How can you tell?
2. Do any of the min/max values in `.describe()` look suspicious?
3. What are the categorical (non-numeric) columns?

*Your answers here:*

1. 
2. 
3. 

---
## 2. Fixing data types

One of the most common — and most silent — data problems is **wrong data types**. If a numeric column is stored as text (`object` dtype), pandas won't compute its mean, won't sort it numerically, and won't plot it correctly. This happens frequently when:
- A CSV has one row with a text value like `"N/A"`, `"<LOD"` (below limit of detection), or `"error"` in an otherwise numeric column
- Numbers were exported with commas (e.g., `"1,234"`)
- Leading/trailing whitespace crept into values

Let's see this in action with a small example.

In [ ]:
# Simulate a common scientific data problem: one bad value in a numeric column
messy_measurements = pd.DataFrame({
    "sample_id": ["A1", "A2", "A3", "A4", "A5"],
    "concentration": ["3.2", "4.1", "<LOD", "5.8", "2.9"],  # "<LOD" = below limit of detection
    "temperature": [22.1, 23.0, 22.8, 21.5, 22.3]
})

print(messy_measurements.dtypes)
print()
messy_measurements

Notice that `concentration` has dtype `object` (text) because of the `"<LOD"` entry. We can't compute a mean on it:

In [ ]:
# This will either error or silently skip the column
try:
    print("Mean concentration:", messy_measurements["concentration"].mean())
except TypeError as e:
    print(f"Error: {e}")

### Fixing it with `pd.to_numeric()`

[`pd.to_numeric()`](https://pandas.pydata.org/docs/reference/api/pandas.to_numeric.html) converts a column to numbers. The `errors='coerce'` parameter turns any non-convertible values into `NaN` (missing) rather than raising an error.

In [ ]:
messy_measurements["concentration"] = pd.to_numeric(
    messy_measurements["concentration"], errors="coerce"
)

print(messy_measurements.dtypes)
print()
print("Mean concentration:", messy_measurements["concentration"].mean())
messy_measurements

The `"<LOD"` value became `NaN`, and the column is now numeric. We can decide how to handle that `NaN` in the next section.

You can also use [`.astype()`](https://pandas.pydata.org/docs/reference/api/pandas.DataFrame.astype.html) for straightforward type conversions (e.g., `df["col"].astype(int)`), but `pd.to_numeric()` is safer when you're not sure the data is clean.

### Exercise 2

The cell below creates a small DataFrame where the `"weight_kg"` column has a problem. Fix it so you can compute the mean weight.

In [ ]:
samples = pd.DataFrame({
    "specimen": ["S1", "S2", "S3", "S4"],
    "weight_kg": ["1.23", "0.98", "  1.10  ", "missing"],  # has whitespace and a text entry
})

# Fix the weight_kg column so it's numeric, then compute the mean
# Your code here


---
## 3. Handling missing data

Missing data is nearly universal in real datasets. In pandas, missing values are represented as **`NaN`** ("Not a Number"). How you handle them depends on *why* they're missing and *how much* data is affected.

### 3a. Detecting missing data

Let's see how much data is missing in the flights dataset, column by column.

In [ ]:
# Count missing values per column
missing_counts = flights.isna().sum()
missing_counts

In [ ]:
# Visualize: which columns have the most missing data?
missing_counts[missing_counts > 0].sort_values().plot(
    kind="barh", color="steelblue"
)
plt.xlabel("Number of missing values")
plt.title("Missing data by column")
plt.tight_layout()

In [ ]:
# What percentage of each column is missing?
missing_pct = (flights.isna().sum() / len(flights) * 100).round(1)
missing_pct[missing_pct > 0].sort_values(ascending=False)

### 3b. Why is data missing?

Before deciding what to do about missing values, it helps to think about **why** they're missing. Statisticians categorize missingness into three types:

| Type | Meaning | Flight example | Impact |
|------|---------|----------------|--------|
| **MCAR** (Missing Completely At Random) | Missingness is unrelated to any variable | A sensor randomly glitched and didn't record air time | Safest to drop — no bias introduced |
| **MAR** (Missing At Random) | Missingness is related to *other observed* variables | Departure delay is missing because the flight was cancelled (and cancellation is recorded) | Dropping may introduce bias; imputation is better |
| **MNAR** (Missing Not At Random) | Missingness is related to *the missing value itself* | Arrival delay is missing because the plane was so late it was diverted — the extreme delays are the ones missing | Most dangerous — any strategy introduces bias |

You don't always know which type you're dealing with, but thinking about it helps you make better decisions.

For more, see [this guide to missing data in Python](https://pandas.pydata.org/pandas-docs/stable/user_guide/missing_data.html).

### 3c. Strategy 1: Dropping rows with missing data

The simplest approach: remove rows that have `NaN` values. Use [`.dropna()`](https://pandas.pydata.org/docs/reference/api/pandas.DataFrame.dropna.html).

In [ ]:
# How many rows do we have before dropping?
print(f"Before: {len(flights):,} rows")

# Drop rows where arr_delay is missing
flights_dropped = flights.dropna(subset=["arr_delay"])

rows_lost = len(flights) - len(flights_dropped)
pct_lost = rows_lost / len(flights) * 100
print(f"After:  {len(flights_dropped):,} rows")
print(f"Lost:   {rows_lost:,} rows ({pct_lost:.1f}%)")

Losing a small percentage of a large dataset is usually fine. But in scientific work, you often have **small** datasets where every sample is precious. That's when imputation becomes important.

> **When to drop:**
> - The percentage of missing data is small (< 5%)
> - You believe the data is MCAR
> - You have plenty of remaining data
>
> **When NOT to drop:**
> - You'd lose a large fraction of your data
> - The missing values follow a pattern (MAR or MNAR)
> - Every sample is expensive to collect

### 3d. Strategy 2: Imputing (filling in) missing values

**Imputation** means replacing `NaN` values with a reasonable substitute. Common strategies:

- **Mean/median** of the column — simple and common
- **A constant** like 0, or a domain-specific value (e.g., "below detection limit")
- **Forward/backward fill** — for time series data, use the previous or next value

Let's try imputing the `air_time` column (which has some missing values) with the median.

In [ ]:
# How many missing values in air_time?
print(f"Missing air_time values: {flights['air_time'].isna().sum():,}")
print(f"Median air_time: {flights['air_time'].median():.0f} minutes")

In [ ]:
# fillna() replaces NaN with the specified value
air_time_filled = flights["air_time"].fillna(flights["air_time"].median())

print(f"Missing before: {flights['air_time'].isna().sum()}")
print(f"Missing after:  {air_time_filled.isna().sum()}")

You can also use **scikit-learn's `SimpleImputer`**, which is useful when you need to apply the same imputation strategy to multiple columns consistently (especially important for machine learning pipelines).

In [ ]:
from sklearn.impute import SimpleImputer

# Create an imputer that fills NaN with the median
imputer = SimpleImputer(strategy="median")

# Fit on the column and transform it
# Note: SimpleImputer expects a 2D array, so we use [[ ]] to select a DataFrame, not a Series
air_time_imputed = imputer.fit_transform(flights[["air_time"]])

print(f"Missing after SimpleImputer: {np.isnan(air_time_imputed).sum()}")

> For more advanced imputation (e.g., using relationships between columns), see scikit-learn's [KNNImputer](https://scikit-learn.org/stable/modules/generated/sklearn.impute.KNNImputer.html) and [IterativeImputer](https://scikit-learn.org/stable/modules/generated/sklearn.impute.IterativeImputer.html).

### Exercise 3

The `dep_delay` column also has missing values.

In [ ]:
# a) How many missing values does dep_delay have?
# Your code here


In [ ]:
# b) Fill in the missing dep_delay values with the median.
#    Then verify there are no more missing values.
# Your code here


In [ ]:
# c) Compare the mean and standard deviation of dep_delay before and after imputation.
#    Did imputation change the distribution much?
# Your code here


---
## 4. Finding and removing duplicates

Duplicate rows can sneak into datasets when:
- Data is exported or appended more than once
- Instruments log the same measurement twice
- Multiple data sources overlap

Duplicates inflate your sample size and can bias results. Use [`.duplicated()`](https://pandas.pydata.org/docs/reference/api/pandas.DataFrame.duplicated.html) to detect them.

In [ ]:
# How many fully duplicated rows are there?
n_duplicates = flights.duplicated().sum()
print(f"Duplicate rows: {n_duplicates:,} out of {len(flights):,}")

In [ ]:
# Let's look at a few duplicated rows
duplicate_mask = flights.duplicated(keep=False)  # keep=False marks ALL copies, not just the extras
flights[duplicate_mask].sort_values(["month", "day", "carrier", "flight"]).head(10)

In [ ]:
# Remove duplicates, keeping the first occurrence
flights_deduped = flights.drop_duplicates()
print(f"Before: {len(flights):,} rows")
print(f"After:  {len(flights_deduped):,} rows")

You can also check for duplicates on a **subset** of columns. For example, should there ever be two flights with the same carrier, flight number, month, and day?

### Exercise 4

Check for duplicate rows using only the columns `carrier`, `flight`, `month`, and `day`. How many are there? What might these represent — are they true duplicates or something else?

In [ ]:
# Your code here


---
## 5. Detecting outliers

An **outlier** is a value that is far outside the expected range. Some outliers are real (an unusually long flight delay during a blizzard) and some are errors (a typo, a sensor glitch, a unit conversion mistake).

The first step is always **detection** — look at the data and ask: "Is this physically possible?"

In [ ]:
# .describe() is a quick way to check for suspicious values
flights[["dep_delay", "arr_delay", "air_time", "distance"]].describe()

Look at the min and max values. Do any of them seem unreasonable?

- Can a departure delay be negative? (Yes — it means the flight left early.)
- Can `air_time` be negative? (No — that would be an error.)
- What about a departure delay of 1,000+ minutes (over 16 hours)?

In [ ]:
# Visualize the distribution of departure delay to spot outliers
fig, axes = plt.subplots(1, 2, figsize=(12, 4))

# Full distribution
axes[0].hist(flights["dep_delay"].dropna(), bins=100, edgecolor="white")
axes[0].set_xlabel("Departure delay (min)")
axes[0].set_ylabel("Count")
axes[0].set_title("Full distribution")

# Zoomed in to the extreme values
extreme = flights[flights["dep_delay"] > 300]
axes[1].hist(extreme["dep_delay"].dropna(), bins=50, edgecolor="white", color="coral")
axes[1].set_xlabel("Departure delay (min)")
axes[1].set_title(f"Delays > 5 hours ({len(extreme):,} flights)")

plt.tight_layout()

In [ ]:
# How many flights have extreme departure delays?
print(f"Delays > 5 hours:  {(flights['dep_delay'] > 300).sum():,}")
print(f"Delays > 12 hours: {(flights['dep_delay'] > 720).sum():,}")
print(f"Max delay:         {flights['dep_delay'].max():.0f} minutes ({flights['dep_delay'].max() / 60:.0f} hours)")

### What to do with outliers?

There's no single right answer — it depends on your goal:

| Strategy | When to use |
|----------|-------------|
| **Keep them** | The extreme values are real and relevant to your analysis |
| **Remove them** | You're confident they're errors, or they'll distort your model |
| **Investigate** | Look at the surrounding data — is there an explanation? |
| **Cap / winsorize** | Replace extreme values with a threshold (e.g., cap at the 99th percentile) |

In scientific work, **document** whatever you decide and **report** how many data points were excluded.

### Exercise 5

Investigate the `air_time` column.

In [ ]:
# a) Are there any flights with air_time <= 0? If so, how many?
# Your code here


In [ ]:
# b) Plot a histogram of air_time. Do the values look reasonable?
# Your code here


In [ ]:
# c) What is the longest flight in the dataset? How many minutes/hours is it?
#    Does this seem plausible for a domestic US flight?
# Your code here


---
## 6. Sanity checks and validation

After cleaning, always verify that your data makes sense. Write checks for things you **know** should be true. If a check fails, something went wrong in your cleaning pipeline.

Python's `assert` statement is useful here — it does nothing if the condition is `True`, but raises an error if it's `False`.

In [ ]:
# Example sanity checks
assert flights["month"].between(1, 12).all(), "Month values should be 1-12"
assert flights["day"].between(1, 31).all(), "Day values should be 1-31"
assert (flights["distance"] > 0).all(), "Distance should be positive"

print("All sanity checks passed!")

In [ ]:
# Check categorical columns for unexpected values
print("Unique origins:", sorted(flights["origin"].unique()))
print(f"Number of unique carriers: {flights['carrier'].nunique()}")
print(f"Number of unique destinations: {flights['dest'].nunique()}")

In scientific work, validation checks might include:
- pH values between 0 and 14
- Temperatures in a physically reasonable range
- Concentrations are non-negative
- No future dates in a historical dataset
- Sample IDs match a known list

### Exercise 6

Write 2-3 additional `assert` statements to validate other columns in the flights data. Think about what values would be physically impossible.

In [ ]:
# Your code here — write assert statements that check for impossible values


---
## 7. Groupby and aggregate (quick overview)

Once your data is clean, a common next step is to **summarize** it by groups. Pandas' [`.groupby()`](https://pandas.pydata.org/docs/reference/api/pandas.DataFrame.groupby.html) lets you split the data by one or more columns and compute statistics on each group.

In [ ]:
# Average departure delay by month
flights.groupby("month")[["dep_delay"]].mean()

In [ ]:
# Multiple statistics at once with .aggregate()
flights[["dep_delay", "arr_delay"]].aggregate(["mean", "median", "min", "max"])

In [ ]:
# Groupby multiple columns
flights.groupby(["month", "origin"])[["dep_delay"]].mean()

### Exercise 7

Which airline (`carrier`) has the highest average departure delay? Sort the results to find out.

In [ ]:
# Your code here


---
## Key Points

- **Always inspect first**: `.info()`, `.describe()`, and `.dtypes` are your starting points.
- **Fix data types early**: use `pd.to_numeric(errors='coerce')` when numeric columns are stored as text.
- **Missing data requires thought**: understand *why* data is missing (MCAR/MAR/MNAR) before deciding to drop or impute.
- **`.dropna()`** removes rows with missing values; **`.fillna()`** replaces them.
- **Check for duplicates** with `.duplicated()` — they're easy to miss and can bias your results.
- **Spot outliers** with `.describe()` and histograms — ask "is this physically possible?"
- **Validate** with `assert` statements after cleaning to catch errors early.
- **Document** every cleaning decision — your future self (and your reviewers) will thank you.

## What's next?

Now that your data is clean, you're ready to start building models! In the next notebook, we'll use **scikit-learn** to preprocess features and train a machine learning model on this flight data.